In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Najafgarh_Delhi_DPCC_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,136.0,147.0,138.0,54.0,70.0,72.0,42.0,NaN,121.0,124.0,354.0,366.0
1,2,290.0,146.0,203.0,58.0,69.0,71.0,60.0,22.0,125.0,128.0,408.0,306.0
2,3,366.0,119.0,151.0,95.0,75.0,101.0,91.0,20.0,122.0,134.0,491.0,246.0
3,4,321.0,150.0,106.0,82.0,50.0,111.0,84.0,52.0,118.0,155.0,393.0,305.0
4,5,256.0,184.0,99.0,123.0,148.0,133.0,62.0,59.0,110.0,145.0,481.0,295.0
5,6,297.0,174.0,128.0,107.0,203.0,64.0,48.0,75.0,NaN,166.0,408.0,272.0
6,7,267.0,219.0,145.0,122.0,110.0,280.0,46.0,73.0,55.0,138.0,388.0,273.0
7,8,267.0,87.0,144.0,141.0,116.0,122.0,NaN,86.0,43.0,125.0,433.0,287.0
8,9,376.0,162.0,102.0,135.0,181.0,108.0,39.0,99.0,26.0,125.0,452.0,278.0
9,10,332.0,122.0,151.0,174.0,229.0,106.0,NaN,114.0,20.0,NaN,236.0,293.0


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    35 non-null     float64
 2   February   33 non-null     float64
 3   March      36 non-null     float64
 4   April      34 non-null     float64
 5   May        37 non-null     float64
 6   June       34 non-null     float64
 7   July       23 non-null     float64
 8   August     35 non-null     float64
 9   September  33 non-null     float64
 10  October    35 non-null     float64
 11  November   35 non-null     float64
 12  December   36 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [8]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,136.0,147.0,138.0,54.0,70.0,72.0,42.000000,80.628571,121.0,124.0,354.0,366.0
1,2,290.0,146.0,203.0,58.0,69.0,71.0,60.000000,22.000000,125.0,128.0,408.0,306.0
2,3,366.0,119.0,151.0,95.0,75.0,101.0,48.826087,20.000000,122.0,134.0,491.0,246.0
3,4,321.0,150.0,106.0,82.0,50.0,111.0,48.826087,52.000000,118.0,155.0,393.0,305.0
4,5,256.0,184.0,99.0,123.0,148.0,133.0,48.826087,59.000000,110.0,145.0,481.0,295.0
